# Notebook 04: User Split (XuetangX)

**Purpose:** Split users into disjoint train / val / test sets. No user appears in two splits.

**Split ratio:** 70 / 15 / 15

**Inputs:** `data/processed/xuetangx/pairs/pairs.parquet`

**Outputs:**
- `data/processed/xuetangx/user_splits/users_{train|val|test}.json`
- `data/processed/xuetangx/pairs/pairs_{train|val|test}.parquet`

In [1]:
# [CELL 04-00] Bootstrap

import json, time, uuid, hashlib
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd
import duckdb

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists(): return p
    raise RuntimeError('PROJECT_STATE.md not found')

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_INTERIM':   REPO_ROOT / 'data' / 'interim',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
}

def cell_start(cid, title, **kw):
    t = time.time()
    print(f'\n[{cid}] {title}')
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    return t

def cell_end(cid, t0, **kw):
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')

def write_json_atomic(path, obj, indent=2):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)

def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def sha256_file(path):
    import hashlib
    h = hashlib.sha256()
    with open(path,'rb') as f:
        while chunk := f.read(1<<20): h.update(chunk)
    return h.hexdigest()

print(f'[CELL 04-00] REPO_ROOT={REPO_ROOT}  done')

[CELL 04-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta  done


In [2]:
# [CELL 04-01] Seed + config

import random
GLOBAL_SEED = 20260107
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)

NOTEBOOK_NAME = '04_user_split_xuetangx'
DATASET       = 'xuetangx'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID  = uuid.uuid4().hex

OUT_DIR = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

DUCKDB_PATH   = PATHS['DATA_INTERIM'] / 'xuetangx.duckdb'
PAIRS_PARQUET = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs' / 'pairs.parquet'

SPLITS_DIR = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'user_splits'
PAIRS_DIR  = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

OUT_USERS_TRAIN  = SPLITS_DIR / 'users_train.json'
OUT_USERS_VAL    = SPLITS_DIR / 'users_val.json'
OUT_USERS_TEST   = SPLITS_DIR / 'users_test.json'
OUT_PAIRS_TRAIN  = PAIRS_DIR / 'pairs_train.parquet'
OUT_PAIRS_VAL    = PAIRS_DIR / 'pairs_val.parquet'
OUT_PAIRS_TEST   = PAIRS_DIR / 'pairs_test.parquet'

# XuetangX split ratio: 70 / 15 / 15 (per CLAUDE.md)
CFG = {
    'notebook': NOTEBOOK_NAME, 'dataset': DATASET, 'seed': GLOBAL_SEED,
    'split': {'train': 0.70, 'val': 0.15, 'test': 0.15},
}
write_json_atomic(CONFIG_PATH, CFG)
for path, obj in [(REPORT_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,
                                   'created_at':datetime.now().isoformat(timespec='seconds'),
                                   'metrics':{},'key_findings':[],'sanity_samples':{},'data_fingerprints':{},'notes':[]}),
                   (MANIFEST_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'artifacts':[]})]: 
    write_json_atomic(path, obj)
META = PATHS['META_REGISTRY']
if not META.exists(): write_json_atomic(META, {'schema_version':1,'runs':[]})
m = read_json(META); m['runs'].append({'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'out_dir':str(OUT_DIR)})
write_json_atomic(META, m)
print(f'[CELL 04-01] run={RUN_TAG}  done')

[CELL 04-01] run=20260408_141441  done


In [3]:
# [CELL 04-02] Load pairs

t0 = cell_start('CELL 04-02', 'Load pairs')

if not PAIRS_PARQUET.exists():
    raise RuntimeError(f'Missing {PAIRS_PARQUET}. Run 03_vocab_pairs_xuetangx.ipynb first.')

pairs_df = pd.read_parquet(PAIRS_PARQUET)
print(f'[CELL 04-02] pairs shape: {pairs_df.shape}')
print(f'[CELL 04-02] users: {pairs_df["user_id"].nunique():,}')
cell_end('CELL 04-02', t0)


[CELL 04-02] Load pairs
[CELL 04-02] pairs shape: (487696, 7)
[CELL 04-02] users: 70,809
[CELL 04-02] elapsed=0.32s  done


In [4]:
# [CELL 04-03] Shuffle users (seeded) and split 70/15/15

t0 = cell_start('CELL 04-03', 'Split users 70/15/15')

unique_users = sorted(pairs_df['user_id'].unique())  # sorted for determinism
n_total = len(unique_users)

rng = np.random.RandomState(GLOBAL_SEED)
shuffled = np.array(unique_users)
rng.shuffle(shuffled)

n_train = int(n_total * CFG['split']['train'])
n_val   = int(n_total * CFG['split']['val'])
n_test  = n_total - n_train - n_val

users_train = shuffled[:n_train].tolist()
users_val   = shuffled[n_train:n_train+n_val].tolist()
users_test  = shuffled[n_train+n_val:].tolist()

print(f'[CELL 04-03] Total users: {n_total:,}')
print(f'[CELL 04-03] Train: {len(users_train):,} ({len(users_train)/n_total:.1%})')
print(f'[CELL 04-03] Val:   {len(users_val):,} ({len(users_val)/n_total:.1%})')
print(f'[CELL 04-03] Test:  {len(users_test):,} ({len(users_test)/n_total:.1%})')

# Disjoint check
assert len(set(users_train) & set(users_val)) == 0, 'train/val overlap'
assert len(set(users_train) & set(users_test)) == 0, 'train/test overlap'
assert len(set(users_val) & set(users_test)) == 0, 'val/test overlap'
print('[CELL 04-03] Disjoint check passed')

cell_end('CELL 04-03', t0, n_train=len(users_train), n_val=len(users_val), n_test=len(users_test))


[CELL 04-03] Split users 70/15/15
[CELL 04-03] Total users: 70,809
[CELL 04-03] Train: 49,566 (70.0%)
[CELL 04-03] Val:   10,621 (15.0%)
[CELL 04-03] Test:  10,622 (15.0%)
[CELL 04-03] Disjoint check passed
[CELL 04-03] n_train=49566
[CELL 04-03] n_val=10621
[CELL 04-03] n_test=10622
[CELL 04-03] elapsed=0.05s  done


In [5]:
# [CELL 04-04] Save user split lists

t0 = cell_start('CELL 04-04', 'Save user lists')
write_json_atomic(OUT_USERS_TRAIN, users_train)
write_json_atomic(OUT_USERS_VAL,   users_val)
write_json_atomic(OUT_USERS_TEST,  users_test)
print(f'[CELL 04-04] Saved {OUT_USERS_TRAIN.name}, {OUT_USERS_VAL.name}, {OUT_USERS_TEST.name}')
cell_end('CELL 04-04', t0)


[CELL 04-04] Save user lists
[CELL 04-04] Saved users_train.json, users_val.json, users_test.json
[CELL 04-04] elapsed=0.02s  done


In [6]:
# [CELL 04-05] Split pairs by user assignment

t0 = cell_start('CELL 04-05', 'Assign pairs to splits')

train_set, val_set, test_set = set(users_train), set(users_val), set(users_test)

pairs_train = pairs_df[pairs_df['user_id'].isin(train_set)].copy()
pairs_val   = pairs_df[pairs_df['user_id'].isin(val_set)].copy()
pairs_test  = pairs_df[pairs_df['user_id'].isin(test_set)].copy()

print(f'[CELL 04-05] Train pairs: {len(pairs_train):,} ({len(pairs_train)/len(pairs_df):.1%})')
print(f'[CELL 04-05] Val pairs:   {len(pairs_val):,} ({len(pairs_val)/len(pairs_df):.1%})')
print(f'[CELL 04-05] Test pairs:  {len(pairs_test):,} ({len(pairs_test)/len(pairs_df):.1%})')

assert len(pairs_train) + len(pairs_val) + len(pairs_test) == len(pairs_df), 'Pair count mismatch'
print('[CELL 04-05] All pairs assigned exactly once')

cell_end('CELL 04-05', t0)


[CELL 04-05] Assign pairs to splits
[CELL 04-05] Train pairs: 344,532 (70.6%)
[CELL 04-05] Val pairs:   69,663 (14.3%)
[CELL 04-05] Test pairs:  73,501 (15.1%)
[CELL 04-05] All pairs assigned exactly once
[CELL 04-05] elapsed=0.11s  done


In [8]:
# [CELL 04-06] Save split pairs + register DuckDB views

t0 = cell_start('CELL 04-06', 'Save split pairs + DuckDB views')

pairs_train.to_parquet(OUT_PAIRS_TRAIN, index=False, compression='zstd')
pairs_val.to_parquet(OUT_PAIRS_VAL, index=False, compression='zstd')
pairs_test.to_parquet(OUT_PAIRS_TEST, index=False, compression='zstd')

for name, path in [('train', OUT_PAIRS_TRAIN), ('val', OUT_PAIRS_VAL), ('test', OUT_PAIRS_TEST)]:
    print(f'[CELL 04-06] Saved pairs_{name}: {path.stat().st_size/(1<<20):.1f} MB')

def esc(p): return str(p).replace("'","''")
con = duckdb.connect(str(DUCKDB_PATH), read_only=False)
for split, path in [('train', OUT_PAIRS_TRAIN), ('val', OUT_PAIRS_VAL), ('test', OUT_PAIRS_TEST)]:
    vname = f'xuetangx_pairs_{split}'
    con.execute(f'DROP VIEW IF EXISTS {vname};')
    con.execute(f"CREATE VIEW {vname} AS SELECT * FROM read_parquet('{esc(path)}')")
    n = con.execute(f'SELECT COUNT(*) FROM {vname}').fetchone()[0]
    print(f'[CELL 04-06] View {vname}: {n:,} rows')
con.close()

# Update report
r = read_json(REPORT_PATH)
r['metrics'] = {'n_users_total': n_total, 'n_users_train': len(users_train),
                 'n_users_val': len(users_val), 'n_users_test': len(users_test),
                 'n_pairs_train': len(pairs_train), 'n_pairs_val': len(pairs_val),
                 'n_pairs_test': len(pairs_test), 'split': CFG['split']}
write_json_atomic(REPORT_PATH, r)

cell_end('CELL 04-06', t0)


[CELL 04-06] Save split pairs + DuckDB views
[CELL 04-06] Saved pairs_train: 4.6 MB
[CELL 04-06] Saved pairs_val: 1.2 MB
[CELL 04-06] Saved pairs_test: 1.3 MB
[CELL 04-06] View xuetangx_pairs_train: 344,532 rows
[CELL 04-06] View xuetangx_pairs_val: 69,663 rows
[CELL 04-06] View xuetangx_pairs_test: 73,501 rows
[CELL 04-06] elapsed=0.63s  done


## Notebook 04 Complete

Split: 70/15/15 — disjoint users across train/val/test (cold-start guarantee).

**Next:** `05_episode_index_xuetangx.ipynb` — build K=5 support, Q=10 query episode index.

**Verify after NB05:** `assert len(episodes_test) == 986`